# Module Speech-to-Text (STT) avec Whisper

Ce notebook implémente la partie **Parole → Texte** du projet *Voix-langue-des-signes*.

Objectif :
- Enregistrer la voix d’un utilisateur à partir du micro.
- Transcrire l’audio en texte (français) à l’aide du modèle Whisper d’OpenAI.
- Préparer une fonction réutilisable (`speech_to_text`) que les autres parties du projet pourront appeler.

Ce notebook est utilisé dans la branche `feature/stt-yohann` du dépôt Git.


## 1. Configuration de `ffmpeg`

Whisper s’appuie sur `ffmpeg` pour charger et décoder les fichiers audio (WAV, MP3, etc.).

Sur notre machine Windows, nous avons :
1. Téléchargé **ffmpeg** depuis le site : https://www.gyan.dev/ffmpeg/builds/
2. Choisi l’archive `ffmpeg-release-essentials.zip`
3. Extrait le contenu dans le dossier : `C:\ffmpeg\ffmpeg-8.0-essentials_build\`
4. Ajouté le chemin `C:\ffmpeg\ffmpeg-8.0-essentials_build\bin` à la variable d’environnement `PATH` de Windows.

Cependant, le kernel Python utilisé par VS Code ne récupère pas toujours correctement le `PATH` système.
Nous ajoutons donc explicitement ce chemin dans le `PATH` du **processus Python** dans ce notebook.


In [2]:
import os
import shutil

# On ajoute le dossier ffmpeg au PATH du processus Python
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"

print("ffmpeg trouvé par Python ? ->", shutil.which("ffmpeg"))


ffmpeg trouvé par Python ? -> C:\ffmpeg\ffmpeg-8.0-essentials_build\bin\ffmpeg.EXE


## 2. Installation des dépendances Python

Les principales bibliothèques utilisées sont :
- `whisper` : modèle de reconnaissance vocale pré-entraîné.
- `sounddevice` : enregistrement audio depuis le micro.
- `scipy` + `numpy` : manipulation et sauvegarde des signaux audio.

Sur cette machine, ces paquets ont été installés via `pip` :

```bash
pip install openai-whisper sounddevice scipy


In [ ]:
# À exécuter seulement si les libs ne sont pas encore installées sur la machine
# !pip install openai-whisper sounddevice scipy --quiet

## 3. Imports et configuration de base

Dans cette cellule, nous :
- importons les modules nécessaires (`whisper`, `sounddevice`, `numpy`, etc.),
- affichons le dossier de travail courant,
- listons les fichiers présents (utile pour vérifier la présence de `mon_audio.wav` après enregistrement).


In [3]:
import whisper
import sounddevice as sd
from scipy.io.wavfile import write
import numpy as np

import os

print("Dossier courant :", os.getcwd())
print("Fichiers présents :", os.listdir())


Dossier courant : d:\Desktop\voix-langue-des-signes\stt
Fichiers présents : ['mon_audio.wav', 'stt_speech_to_text.ipynb']


## 4. Chargement du modèle Whisper

Nous utilisons le modèle **`small`** de Whisper, qui offre un bon compromis entre :
- temps de calcul (raisonnable sur CPU),
- qualité de la transcription.

Remarque :
- Le warning `FP16 is not supported on CPU; using FP32 instead` est **normal** sur CPU :
  cela signifie simplement que le modèle utilisera des flottants 32 bits au lieu de 16 bits.


In [4]:
model = whisper.load_model("small")
print("Modèle Whisper chargé.")


Modèle Whisper chargé.


## 5. Enregistrement audio depuis le micro

Cette cellule enregistre la voix de l'utilisateur via le microphone.

Paramètres :
- `filename` : nom du fichier audio de sortie (`mon_audio.wav`).
- `duration` : durée de l'enregistrement (en secondes).
- `fs` : fréquence d'échantillonnage (16 kHz).

Le fichier est sauvegardé dans le **dossier courant** du notebook, ici :
`d:\\Desktop\\voix-langue-des-signes\\stt`

Après enregistrement, nous affichons à nouveau la liste des fichiers pour vérifier la présence de `mon_audio.wav`.


In [5]:
def record_audio(filename="mon_audio.wav", duration=5, fs=16000):
    print("Dossier courant :", os.getcwd())
    print("Enregistrement… Parle maintenant.")
    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype="float32")
    sd.wait()
    audio_int16 = np.int16(audio * 32767)
    write(filename, fs, audio_int16)
    print("Enregistrement terminé :", filename)
    print("Fichiers présents :", os.listdir())

# Enregistrement d'un exemple
record_audio("mon_audio.wav", duration=5)


Dossier courant : d:\Desktop\voix-langue-des-signes\stt
Enregistrement… Parle maintenant.
Enregistrement terminé : mon_audio.wav
Fichiers présents : ['mon_audio.wav', 'stt_speech_to_text.ipynb']


## 6. Transcription de l'audio en texte

Cette cellule définit une fonction `speech_to_text` qui :

1. Vérifie que le fichier audio existe réellement (pour éviter les erreurs de type `FileNotFoundError`).
2. Appelle `model.transcribe(...)` de Whisper avec :
   - le chemin du fichier (`audio_path`),
   - la langue cible (`language="fr"`).
3. Retourne uniquement le champ `["text"]` (transcription finale).

Nous appliquons ensuite cette fonction sur `mon_audio.wav` pour afficher la transcription reconnue.


In [6]:
def speech_to_text(audio_path, language="fr"):
    # Vérification défensive : le fichier existe-t-il ?
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"Le fichier {audio_path} est introuvable dans {os.getcwd()}")
    
    print("Transcription de :", audio_path)
    result = model.transcribe(audio_path, language=language)
    return result["text"]

# Test sur l'enregistrement précédent
texte = speech_to_text("mon_audio.wav")
print("Texte reconnu :", texte)


Transcription de : mon_audio.wav


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Texte reconnu :  Bonjour ça va ?


## 7. Récapitulatif détaillé : comment nous avons fait fonctionner la transcription

Cette section résume les étapes nécessaires pour faire fonctionner la transcription *parole → texte* avec Whisper dans notre environnement Windows + VS Code.

1. **Installation de ffmpeg**
   - Téléchargement de l’archive `ffmpeg-release-essentials.zip` depuis le site : https://www.gyan.dev/ffmpeg/builds/
   - Extraction du contenu dans `C:\ffmpeg\ffmpeg-8.0-essentials_build\`.
   - Vérification de la présence de `ffmpeg.exe` dans `C:\ffmpeg\ffmpeg-8.0-essentials_build\bin`.

2. **Ajout de ffmpeg au PATH**
   - Ajout de `C:\ffmpeg\ffmpeg-8.0-essentials_build\bin` dans la variable d’environnement `PATH` de Windows.
   - Vérification dans PowerShell que `ffmpeg` est bien reconnu via :
     ```powershell
     ffmpeg -version
     ```

3. **Problème rencontré dans le notebook VS Code**
   - Dans le kernel Python du notebook, appel à Whisper → erreur :
     `FileNotFoundError: [WinError 2] Le fichier spécifié est introuvable`.
   - Ce message ne concernait pas le `.wav` (qui existait bien dans le dossier du notebook), mais l'exécutable `ffmpeg` qui n'était pas trouvé par le **processus Python**.

4. **Correction dans le notebook**
   - Ajout explicite de `C:\ffmpeg\ffmpeg-8.0-essentials_build\bin` au `PATH` du processus dans le notebook :
     ```python
     os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"
     ```
   - Vérification que Python trouve bien `ffmpeg` :
     ```python
     import shutil
     print(shutil.which("ffmpeg"))
     ```
   - Une fois cela corrigé, Whisper a pu lire le fichier `mon_audio.wav` et la transcription a fonctionné.

5. **Pipeline final fonctionnel**
   - Enregistrement audio avec `sounddevice` dans `mon_audio.wav`.
   - Chargement du modèle Whisper `small`.
   - Appel de `model.transcribe("mon_audio.wav", language="fr")`.
   - Récupération de la transcription via `result["text"]`.

Conclusion :  
Nous disposons désormais d'un module STT fiable, capable de transformer un enregistrement audio micro en texte, et intégrable à la suite du pipeline (*texte → gloss → animations en langue des signes*).


## Améliorations prévues dans la V2

Dans la version 2 du module STT, nous ajouterons :

- une boucle **semi temps réel** avec des segments de 2 secondes,
- un filtrage des segments silencieux (détection d'énergie),
- un comportement plus interactif : texte affiché au fur et à mesure,
- puis, dans une version encore suivante, un **arrêt vocal** (mot-clé "stop").

La V1 sert donc de base minimale : enregistrement + transcription d'un seul segment.
